Obtain ID's of records that have zero subjects.  
Then free to experiment with methods to enrich their metadata.

In [56]:
NO_RECORDS = 150

In [57]:
from datetime import datetime, timezone
from json import JSONDecodeError
from logging.config import dictConfig
from typing import Any, Optional
import psycopg
from psycopg import errors as psycopg_errors
from psycopg.rows import dict_row
from config.logging_config import LOGGING_CONFIG
from config.postgres_config import PostgresConfig
from tasks import transform_batch, add_file_metadata
import os
from fastapi import FastAPI, Query, HTTPException
import logging
from pydantic import BaseModel, Field
from utils.queue_utils import HarvestEventQueue, detect_identifier_type

from lxml import etree as ET
# API calls
import requests


# added for these experiments
from utils.enrichment_methods import get_doi, clean_doi, enrich_subjects_from_openaire, merge_subjects_from_new_method

In [58]:
dictConfig(LOGGING_CONFIG)
logger = logging.getLogger(__name__)

In [59]:
user = os.getenv("POSTGRES_USER")
passwd = os.getenv("POSTGRES_PASSWORD")
host = os.getenv("POSTGRES_ADDRESS", "127.0.0.1")


if host == '':
    host = "127.0.0.1"
    user = os.getenv("POSTGRES_ADMIN")

connection_params = {
    "user" : user,
    "password" : passwd,
    "dbname" : "datasetdb",
    "port" : 5432,
    "host" : host
}

In [60]:
query_records = """
            -- Exploring subjects from record raw metadata

-- Every xpath defines a potential XML structure with corresponding subject tag placement.
-- If raw_metadata conforms to ANY of these xpaths, subjects will be concatenated <subject> tags. (removing array_to_string() will return arrays)
-- If raw_metadata doesn't conform to any path (and probably has no subject tags), subjects will be ''


WITH query AS (
    SELECT
        records.*,
        array_to_string(
                xpath('oai:record/oai:metadata/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oaidc:oai_datacite/oaidc:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/},
                      {oaidc, http://schema.datacite.org/oai/oai-1.1/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oai:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
                      {oai, http://www.openarchives.org/OAI/2.0/}}'
                )::text[] ||
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
				-- note the 3 in datacite
                    '{{datacite, http://datacite.org/schema/kernel-3},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[] ||
				-- same path as previous but uses kernel-4 namespace
				xpath('oai:record/oai:metadata/oad:oai_datacite/oad:payload/datacite:resource/datacite:subjects/datacite:subject/text()', raw_metadata,
                    '{{datacite, http://datacite.org/schema/kernel-4},
					  {oai, http://www.openarchives.org/OAI/2.0/},
     				  {oad, http://schema.datacite.org/oai/oai-1.0/}}'
                )::text[]
				, ', '
        ) AS subjects
    FROM public.records
)
SELECT 
    id,
    title,
    raw_metadata,
    subjects
FROM query
WHERE query.subjects = ''
ORDER BY id
OFFSET 100
LIMIT %s
;
"""

In [61]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_records, (NO_RECORDS,))
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_titles = [str(row["title"]) for row in  results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192', 'title': 'Sine 2020 Data Collection', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/103224192</identifier>\n    <datestamp>2019-05-20T14:11:55.780Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB1790346-5</identifier>\n          <alter

In [62]:
record_titles

['Sine 2020 Data Collection',
 'Sine 2020 Data Collection',
 'Sine 2020 Data Collection',
 'Sine 2020 Data Collection',
 'Neutron Training Course - Direct NVS',
 'Neutron Training Course - Direct NVS',
 'Helium3 Cell materials test',
 'IBEX Testing',
 'CRISP membrane diffraction Tests',
 'NTC',
 'Comissioning',
 'Startup and Calibration',
 'BrightnESS2 round1',
 'Surfactant adsorption at the air/water interface',
 'Commissioning New ULT Kit',
 'Commissioning New ULT Kit',
 'Thermodifferential Neutron Transmission (TNT): Pilot Study on a Photovoltaic Material',
 'Study of momentum distributions of Mo in MoO3',
 'Study of momentum distribution of Nd in Nd2O3',
 'Vertical Mode comissioning and Xpress',
 'Commissioning',
 'Quasi-elastic excitation in Kondo system CeBi2',
 'NTC',
 'NTC',
 'Xpress Access on Polaris Cycle 2019/1: XB 1890367, 1990067-71, 1990109-12, 1990131-34, 1990139-43, 1990149-50, 1990162, 1990164-65',
 'In-situ calorimeter comissioning',
 'Polaris Calibration - Cycles 201

In [63]:
record_raw_metadata[2]

'<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/103224220</identifier>\n    <datestamp>2019-05-20T15:52:50.485Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB1790346-6</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>1790346</alternateIdentifier>\n      

In [64]:
get_doi(record_raw_metadata[2])

'10.5286/isis.e.rb1790346-6'

# DOI check through bigger repositories  (#1)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Seaches thru OpenAlex or Openaire - apis?
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

OpenAIRE API
- they're officialy moving to a Graph API in May 2026
https://api.openaire.eu/graph/swagger-ui/index.html

In [65]:
import logging
logging.getLogger("urllib3").setLevel(logging.WARNING)

Old methods, DOI-by-DOI. outdated

In [66]:
doi = get_doi(record_raw_metadata[0]) 

if not doi:
    print("NO DOI")
else:
    print(enrich_subjects_from_openaire(doi))

['RLI838', 'RLI904', 'PG002', 'Technique Development', 'cell', '+', 'Construction Engineering and Architecture', 'RLI908', 'Sine2020', 'Cd', 'RAVIL', 'Engineering', 'RLI882', 'Engineering Sciences', 'Materials', 'STFC ISIS Neutron and Muon Data']


In [67]:
# get only records that have a DOI
ix_has_doi = []

for i in range(0, len(record_raw_metadata)):
    rec = get_doi(record_raw_metadata[i])
    if rec:
        ix_has_doi.append(i)

print(ix_has_doi)

[0, 1, 2, 3, 48, 95, 141, 142, 143]


Batch methods

In [68]:
id_to_enrich = [record_ids[i] for i in ix_has_doi]
doi_to_enrich = [get_doi(record_raw_metadata[i]) for i in ix_has_doi]

print(id_to_enrich)
print(doi_to_enrich)

['0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224281', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/109728434', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/110995124', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/110995129', '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/110995131']
['10.5286/isis.e.rb1790346-5', '10.5286/isis.e.rb1790346-4', '10.5286/isis.e.rb1790346-6', '10.5286/isis.e.rb1790346-7', '10.5286/isis.e.rb1670547-3', '10.5286/isis.e.rb1990292-1', '10.5286/isis.e.rb1890354-2', '10.5286/isis.e.rb1510265-2', '10.5286/isis.e.rb1920617-3']


In [69]:
print(NO_RECORDS)
print(len(doi_to_enrich))

150
9


In [70]:
from utils.enrichment_methods import _openaire_batch_doi_resolve, _opeanaire_get_doi_subjects, connect_ids_with_subjects_by_doi, batch_enrich_subjects_from_openaire

api_result = _openaire_batch_doi_resolve(doi_to_enrich)

print(api_result[:3])

[{'authors': None, 'openAccessColor': None, 'publiclyFunded': None, 'type': 'dataset', 'language': {'code': 'und', 'label': 'Undetermined'}, 'countries': None, 'subjects': [{'subject': {'scheme': 'keyword', 'value': 'STFC ISIS Neutron and Muon Data'}, 'provenance': None}], 'mainTitle': 'Spin wave investigation of Ba2MnTeO6', 'subTitle': None, 'descriptions': None, 'publicationDate': '2019-01-01', 'publisher': None, 'embargoEndDate': None, 'sources': None, 'formats': None, 'contributors': None, 'coverages': None, 'bestAccessRight': None, 'container': None, 'documentationUrls': None, 'codeRepositoryUrl': None, 'programmingLanguage': None, 'contactPeople': None, 'contactGroups': None, 'tools': None, 'size': None, 'version': None, 'geoLocations': None, 'id': 'doi_________::43d719dbc7e664aad6669a3f64962a06', 'originalIds': ['50|datacite____::43d719dbc7e664aad6669a3f64962a06', '10.5286/isis.e.rb1890354-2'], 'pids': [{'scheme': 'doi', 'value': '10.5286/isis.e.rb1890354-2'}], 'dateOfCollection

In [71]:
doi_subjects = _opeanaire_get_doi_subjects(api_result)
doi_subjects

[{'doi': '10.5286/isis.e.rb1890354-2',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-1',
  'subjects': ['RLI838',
   'RLI904',
   'PG002',
   'Technique Development',
   'cell',
   '+',
   'Construction Engineering and Architecture',
   'RLI908',
   'Sine2020',
   'Cd',
   'RAVIL',
   'Engineering',
   'RLI882',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-6',
  'subjects': ['RLI838',
   'RLI904',
   'PG002',
   'Technique Development',
   'cell',
   '+',
   'Construction Engineering and Architecture',
   'RLI908',
   'Sine2020',
   'Cd',
   'RAVIL',
   'Engineering',
   'RLI882',
   'Engineering Sciences',
   'Materials',
   'STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-7',
  'subjects': ['RLI838',
   'RLI904',
   'PG002',
   'Technique Development',
   'cell',
   '+',
   'Construction Engineering and Architecture',
   'RLI908',
   'Sine2020',
   

In [72]:
id_dois = []
for ix in range(len(id_to_enrich)):
    id_dois.append(
        {
            'id': id_to_enrich[ix],
            'doi': doi_to_enrich[ix].lower(),
        }
    )
id_dois

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192',
  'doi': '10.5286/isis.e.rb1790346-5'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211',
  'doi': '10.5286/isis.e.rb1790346-4'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220',
  'doi': '10.5286/isis.e.rb1790346-6'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224281',
  'doi': '10.5286/isis.e.rb1790346-7'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': '10.5286/isis.e.rb1670547-3'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/109728434',
  'doi': '10.5286/isis.e.rb1990292-1'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/110995124',
  'doi': '10.5286/isis.e.rb1890354-2'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/110995129',
  'doi': '10.5286/isis.e.rb1510265-2'},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/110995131',
  'doi': '10.5286/isis.e.rb1920617-3'}]

In [73]:
enriched_list = connect_ids_with_subjects_by_doi(id_dois, doi_subjects)
enriched_list

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192',
  'doi': '10.5286/isis.e.rb1790346-5',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211',
  'doi': '10.5286/isis.e.rb1790346-4',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220',
  'doi': '10.5286/isis.e.rb1790346-6',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engin

In [74]:
# Only succesfully enriched IDs and subjects.
# Could be useful for comparing how many ids we sent to be enriched vs
# how many actually got new subjects
[el for el in enriched_list if el['enriched_subjects']]

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192',
  'doi': '10.5286/isis.e.rb1790346-5',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211',
  'doi': '10.5286/isis.e.rb1790346-4',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220',
  'doi': '10.5286/isis.e.rb1790346-6',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engin

In [75]:
# All-in-one function

batch_enrich_subjects_from_openaire(id_dois)

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192',
  'doi': '10.5286/isis.e.rb1790346-5',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211',
  'doi': '10.5286/isis.e.rb1790346-4',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220',
  'doi': '10.5286/isis.e.rb1790346-6',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engin

In [76]:
# Comparison: Number of initial records vs number of records that have gained subjects via this method
enriched = batch_enrich_subjects_from_openaire(id_dois)
print(len([el['doi'] for el in enriched]))
print(len([el['doi'] for el in enriched if el['enriched_subjects']]))

9
9


In [77]:
# Taking a look at the metadata initially obtained from DB, subset of few records
full_info_enriched = [el for el in enriched if el['enriched_subjects']]

first_enriched = full_info_enriched[:10]

In [78]:
# indexes that have been enriched
ix_original = [record_ids.index(rec['id']) for rec in first_enriched]
ix_original

[0, 1, 2, 3, 48, 95, 141, 142, 143]

In [79]:
first_enriched

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192',
  'doi': '10.5286/isis.e.rb1790346-5',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211',
  'doi': '10.5286/isis.e.rb1790346-4',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engineering and Architecture',
   'Engineering',
   'Engineering Sciences',
   'Materials',
   'PG002',
   'RAVIL',
   'RLI838',
   'RLI882',
   'RLI904',
   'RLI908',
   'STFC ISIS Neutron and Muon Data',
   'Sine2020',
   'Technique Development',
   'cell'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220',
  'doi': '10.5286/isis.e.rb1790346-6',
  'enriched_subjects': {'+',
   'Cd',
   'Construction Engin

In [80]:
[record_raw_metadata[i] for i in ix_original]

['<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">\n  <header>\n    <identifier>oai:icat.isis.stfc.ac.uk:inv/103224192</identifier>\n    <datestamp>2019-05-20T14:11:55.780Z</datestamp>\n    <setSpec>doiInvs</setSpec>\n  </header>\n  <metadata>\n    <oai_datacite xmlns="http://schema.datacite.org/oai/oai-1.1/" xsi:schemaLocation="http://schema.datacite.org/oai/oai-1.1/ http://schema.datacite.org/oai/oai-1.1/oai.xsd">\n      <schemaVersion>4.3</schemaVersion>\n      <datacentreSymbol>BL.ISIS</datacentreSymbol>\n      <payload>\n        <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 http://schema.datacite.org/meta/kernel-4.3/metadata.xsd">\n          <identifier identifierType="DOI">https://doi.org/10.5286/ISIS.E.RB1790346-5</identifier>\n          <alternateIdentifiers alternateIdentifierType="RB">\n            <alternateIdentifier>1790346</alternateIdentifier>\n     

## OpenAlex API
https://developers.openalex.org/
https://developers.openalex.org/guides/authentication

Get multiple works by DOI

use | between DOIs, up to 100 DOIs at once!!
check out how the results look then.

https://api.openalex.org/works?filter=doi:https://doi.org/10.1234/a|https://doi.org/10.1234/b|https://doi.org/10.1234/c&per_page=100



Use the select parameter to return only specific fields:
https://api.openalex.org/works?select=id,doi,display_name

In [81]:
OpenAlex_API_key = os.getenv("OPENALEX_API_KEY")

In [82]:
OPENALEX_API_URL = "https://api.openalex.org/works/"

doi = '10.15151/esrf-es-1473312981'

# We have to append the ID directly in the path
query_url = OPENALEX_API_URL + "https://doi.org/" + doi

response = requests.get(
    query_url,
    params={"api_key": OpenAlex_API_key}, 
    timeout=30
)

data = response.json()


In [83]:
oa_DOIS = [
    '10.15151/esrf-es-1001518064',
    '10.15151/esrf-es-2323561777',
    '10.15151/esrf-es-2323561778',
    '10.15151/esrf-es-2323561877',
    'https://doi.org/10.5286/ISIS.E.100051344',
    '10.34894/AUUSS2',
    '10.17026/DANS-28H-C53V',
    '10.34894/VLSN1K',
    '10.15151/ESRF-ES-1030397492',
    '10.17026/AR/DMHU53',
    '10.17026/AR/GHYQTO',
    '10.17026/DANS-ZU2-HF3N',
    '10.17026/DANS-ZTM-7SA5',
    '10.17026/DANS-ZUK-HN2Z',
    'https://doi.org/10.1016/j.paid.2021.110888',
    '10.23662/fors-ds-686-1',
    '10.1056/NEJMoa2034577',
    '10.1056/NEJMoa2021436',
    '10.1056/NEJMoa1511939',
    '10.1056/NEJMoa1800389'
]

the API result object can have a result for a given DOI, but that result doesn't contain any keywords = doesn't contribute to enrichment
Hence the len(api_result) can be larger than len(enriched_dois)

Additionaly if we set a high keyword score threshold, a certain result MIGHT have keywords but none are relevant enough - that will also not contribute to enrichment

In [84]:
from utils.enrichment_methods import batch_enrich_subjects_from_openalex

enriched_results = batch_enrich_subjects_from_openalex(
    id_dois,
    OpenAlex_API_key,
    score_treshold = 0.6
)

enriched_results = [el for el in enriched_results if el['enriched_subjects']]
print(len(enriched_results))
enriched_results

0


[]

In [85]:
# Showcase  - manually input DOIs

test_dois = [
    {'id': "testid1",
     'doi': '10.1016/j.paid.2021.110888'},
    {'id': "testid2",
     'doi': '10.23662/fors-ds-686-1'}
]

enriched_results = batch_enrich_subjects_from_openalex(
    test_dois,
    OpenAlex_API_key,
    score_treshold = 0.7
)

enriched_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

In [86]:
# Compare openalex with openaire:
oaire_results = batch_enrich_subjects_from_openaire(test_dois)
oaire_results

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

In [87]:
merge_subjects_from_new_method(enriched_results, oaire_results)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': {'05 social sciences',
   '0501 psychology and cognitive sciences',
   '10. No inequality',
   'Conscientiousness',
   'Flow Experience in Various Fields',
   'Mind wandering and attention',
   'Psychology',
   'Sport Psychology and Performance'}},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': {'10. No inequality',
   '3. Good health',
   '4. Education',
   '8. Economic growth'}}]

# DOI resolver method  (#2)
Take an XML ETree.
Searches for identifier with identifierType DOI.
    - what if there are multiple, and one is under "related works"?
Talks to an API such as DataCite or SignPosting.
Data available? N -> end
Y ->
    search for fields: subject, topic, field, discipline, keyword

In [88]:
## DATACITE REST API

DATACITE_API_URL = "https://api.datacite.org/dois"

DataCite provides an additional "enriched" option for fetching doi-by-doi, could be interesting.

In [89]:
from utils.enrichment_methods import batch_enrich_subjects_from_datacite, _datacite_batch_doi_resolve, _datacite_get_doi_subjects

In [90]:
DC_res = _datacite_batch_doi_resolve(doi_to_enrich)

In [91]:
DC_res

[{'id': '10.5286/isis.e.rb1790346-5',
  'type': 'dois',
  'attributes': {'doi': '10.5286/isis.e.rb1790346-5',
   'prefix': '10.5286',
   'suffix': 'isis.e.rb1790346-5',
   'identifiers': [],
   'alternateIdentifiers': [],
   'creators': [{'name': 'Mr Mark Kibble',
     'nameType': 'Personal',
     'affiliation': [],
     'nameIdentifiers': []}],
   'titles': [{'title': 'Sine 2020 Data Collection'}],
   'publisher': 'ISIS Facility',
   'container': {'type': 'DataRepository',
    'identifier': '10.5286/ISIS.E.RB1790346',
    'identifierType': 'DOI'},
   'publicationYear': 2018,
   'subjects': [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
   'contributors': [],
   'dates': [{'date': '2018', 'dateType': 'Issued'}],
   'language': None,
   'types': {'ris': 'DATA',
    'bibtex': 'misc',
    'citeproc': 'dataset',
    'schemaOrg': 'Dataset',
    'resourceTypeGeneral': 'Dataset'},
   'relatedIdentifiers': [{'relationType': 'IsPartOf',
     'relatedIdentifier': '10.5286/isis.e

In [92]:
[el['attributes']['subjects'] for el in DC_res]

[[{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}],
 [{'lang': 'en', 'subject': 'STFC ISIS Neutron and Muon Data'}]]

In [93]:
slist = [el['attributes']['subjects'] for el in DC_res]

[[el.get('subject') for el in sublist] for sublist in slist]

[['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data'],
 ['STFC ISIS Neutron and Muon Data']]

In [94]:
_datacite_get_doi_subjects(DC_res)

[{'doi': '10.5286/isis.e.rb1790346-5',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-4',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-6',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1790346-7',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1670547-3',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1990292-1',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1890354-2',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1510265-2',
  'subjects': ['STFC ISIS Neutron and Muon Data']},
 {'doi': '10.5286/isis.e.rb1920617-3',
  'subjects': ['STFC ISIS Neutron and Muon Data']}]

In [95]:
DC_enriched = batch_enrich_subjects_from_datacite(id_dois)

In [96]:
[el for el in DC_enriched if el['enriched_subjects']]

[{'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224192',
  'doi': '10.5286/isis.e.rb1790346-5',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224211',
  'doi': '10.5286/isis.e.rb1790346-4',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224220',
  'doi': '10.5286/isis.e.rb1790346-6',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/103224281',
  'doi': '10.5286/isis.e.rb1790346-7',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/105604808',
  'doi': '10.5286/isis.e.rb1670547-3',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a-4a96-ac2b-a3f6a1d9f964::inv/109728434',
  'doi': '10.5286/isis.e.rb1990292-1',
  'enriched_subjects': {'STFC ISIS Neutron and Muon Data'}},
 {'id': '0725037e-553a

In [97]:
len([el for el in DC_enriched if el['enriched_subjects']])

9

In [98]:
# All in one method:
from utils.enrichment_methods import batch_enrich_subjects_from_datacite
batch_enrich_subjects_from_datacite(test_dois)

[{'id': 'testid1',
  'doi': '10.1016/j.paid.2021.110888',
  'enriched_subjects': set()},
 {'id': 'testid2',
  'doi': '10.23662/fors-ds-686-1',
  'enriched_subjects': set()}]

# Related DOIs  
Extract related identifiers that are also DOIs. If they are in the database, use their subjects. If not, use DOI-resolving methods on them.

Note:
a DOI must start with 10. prefix. I've seen some related DOIs that start with 0.  , if I add a 1 to the start it resolves properly.

## SQL query to see records with related DOIs   
```
--- get records with related identifiers of any tipe
WITH query AS (
    SELECT
        raw_metadata,
		doi as record_doi,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 1000;
```

47k records have SOME relatedIdentifiers - lots of them ISSN or proprietary ids

In [99]:
# Let's try out some records with related DOIs, for testing we won't limit ourselves to records with no raw_subjects.
# This will get 100 records with some related DOIs

query_related_dois = """
WITH query AS (
    SELECT
        id,
        raw_metadata,
		doi as record_doi,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects,
        xpath($$(//*[local-name()='relatedIdentifier'][@relatedIdentifierType='DOI'])/text()$$, raw_metadata)::text[] AS related
    FROM public.records
)
SELECT *
FROM query
WHERE array_length(related, 1) > 0
LIMIT 100;
"""

In [100]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_related_dois)
        
        results = cur.fetchall()
        print(results)

        record_ids = [str(row["id"]) for row in results]
        record_subjects = [row["subjects"] for row in results]
        record_raw_metadata = [str(row["raw_metadata"]) for row in  results]
        record_related_dois = [row["related"] for row in results]


except Exception as e:
    logger.exception("Failed to fetch harvest runs")
    raise HTTPException(status_code=500, detail=str(e))

[{'id': '470e368d-eeae-46a3-8730-b2ff6afe37cf::10521', 'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>10521</identifier>\n    <datestamp>2026-01-12T10:27:03Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.34965/i10521</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Organizational">Alessandro MANCINI</datacite:creatorName>\n        </datacite:creator>\n      </datacite:creators>\n      <datacite:titles>\n        <datacite:title>20220427</datacite:title>\n      </datacite:titles>\n      <datacite:publisher>Elettra Sincrotrone Trieste</datacite:pu

In [101]:
results

[{'id': '470e368d-eeae-46a3-8730-b2ff6afe37cf::10521',
  'raw_metadata': '<record xmlns="http://www.openarchives.org/OAI/2.0/" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance" xmlns:datacite="http://datacite.org/schema/kernel-4">\n  <header>\n    <identifier>10521</identifier>\n    <datestamp>2026-01-12T10:27:03Z</datestamp>\n  </header>\n  <metadata>\n    <resource xmlns="http://datacite.org/schema/kernel-4" xsi:schemaLocation="http://datacite.org/schema/kernel-4 https://schema.datacite.org/meta/kernel-4.6/metadata.xsd">\n      <datacite:identifier identifierType="DOI">10.34965/i10521</datacite:identifier>\n      <datacite:creators>\n        <datacite:creator>\n          <datacite:creatorName nameType="Organizational">Alessandro MANCINI</datacite:creatorName>\n        </datacite:creator>\n      </datacite:creators>\n      <datacite:titles>\n        <datacite:title>20220427</datacite:title>\n      </datacite:titles>\n      <datacite:publisher>Elettra Sincrotrone Trieste</datacite:

In [102]:
# For a specific DOI set, search our datasetdb and retrieve subjects if any.
# Maybe a record references/is part of another dataset so the subjects are relevant too.

# This query works for a list of DOIs -  if a record has multiples

query_subjects_by_doi = """
WITH query AS (
    SELECT
    id,
        ARRAY(
            SELECT jsonb_array_elements(datacite_json->'subjects')->>'subject'
        ) AS subjects
    FROM public.records
)
SELECT subjects
FROM query
WHERE query.id = ANY(%s)
LIMIT 100;
"""

In [103]:
# Some related dois link to just https://doi.org/

# We'll first apply the cleaning, then drop the empty strings

record_related_dois = [[clean_doi(doi) for doi in sublist if clean_doi(doi)] for sublist in record_related_dois]



In [104]:
record_related_dois

[['10.34965/i10521'],
 [],
 ['10.34965/i9362'],
 ['10.34965/i10487'],
 ['10.34965/i10574'],
 ['10.34965/i9999'],
 ['10.34965/i10332'],
 ['10.34965/i10129'],
 ['10.34965/i9728'],
 ['10.34965/i9417'],
 ['10.34965/i10216'],
 ['10.34965/i10195'],
 ['10.34965/i7734'],
 ['10.34965/i7897'],
 ['10.34965/i10245'],
 ['10.34965/i10915'],
 ['10.34965/i61060'],
 ['10.34965/i10201'],
 ['10.34965/i32619'],
 [],
 ['10.34965/i8083'],
 ['10.34965/i60512'],
 ['10.34965/i10640'],
 ['10.34965/i9834'],
 ['10.34965/i8081'],
 ['10.34965/i10862'],
 ['10.34965/i10594'],
 ['10.34965/i7460'],
 ['10.34965/i7674'],
 ['10.34965/i8074'],
 ['10.34965/i8131'],
 ['10.34965/i8932'],
 ['10.34965/i7699'],
 ['10.34965/i6881'],
 ['10.34965/i8082'],
 ['10.34965/i9612'],
 ['10.34965/i8916'],
 ['10.34965/i10604'],
 ['10.34965/i10001'],
 ['10.34965/i9442'],
 ['10.34965/i10504'],
 ['10.34965/i10160'],
 ['10.34965/i10158'],
 ['10.34965/i10264'],
 ['10.34965/i8193'],
 ['10.34965/i9489'],
 ['10.34965/i10016'],
 ['10.34965/i8808'],
 

In [105]:
first_related_doi = record_related_dois[0]
first_related_doi

['10.34965/i10521']

In [106]:
try:
    with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
        cur = conn.cursor()
        
        cur.execute(query_subjects_by_doi, (first_related_doi,))
        
        record_res = cur.fetchall()
except Exception as e:
    print(f"Query error: {e}")
        

record_res

[]

In [107]:
# Go through all test cases

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]

            # If there are no related DOIs remaining after cleaning, no subjects will be enriched
            if not related_dois:
                related_subjects.append(set())
                continue
            
            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            related_subjects.append(record_res)
    except:
        print(f"Query error: {e}")
        

In [108]:
# No hits
[res for res in related_subjects if res]

[]

In [109]:
# Extra: go through our DB
# and ALSO retrieve all subjects from openaire and openalex for those individual DOIs

related_subjects = []

for i in range(len(record_ids)):
    try:
        with psycopg.connect(**connection_params, row_factory=dict_row) as conn:
            cur = conn.cursor()
            related_dois = record_related_dois[i]

            if not related_dois:
                related_subjects.append(set())
                continue

            cur.execute(query_subjects_by_doi, (related_dois,))
            
            record_res = cur.fetchall()
            record_res = set(record_res)
            # We need an id-doi pairing for each doi, we'll make fake ids since we won't use them for joining later
            id_dois = [{'id': str(i), 'doi': doi,} for i, doi in enumerate(related_dois)]
            if (i < 10):
                print(id_dois)

            openaire = batch_enrich_subjects_from_openaire(id_dois)
            openaire_subjects = set().union(*(el["enriched_subjects"] for el in openaire))
            
            openalex = batch_enrich_subjects_from_openalex(id_dois, OpenAlex_API_key)
            openalex_subjects = set().union(*(el["enriched_subjects"] for el in openalex))

            # Union of all three
            record_res = record_res | openaire_subjects | openalex_subjects

            related_subjects.append(record_res)
    except Exception as e:
        print(f"Query error: {e}")
        

[{'id': '0', 'doi': '10.34965/i10521'}]
[{'id': '0', 'doi': '10.34965/i9362'}]
[{'id': '0', 'doi': '10.34965/i10487'}]
[{'id': '0', 'doi': '10.34965/i10574'}]
[{'id': '0', 'doi': '10.34965/i9999'}]
[{'id': '0', 'doi': '10.34965/i10332'}]
[{'id': '0', 'doi': '10.34965/i10129'}]
[{'id': '0', 'doi': '10.34965/i9728'}]
[{'id': '0', 'doi': '10.34965/i9417'}]


In [110]:
related_subjects

[set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set(),
 set()]